# Convert selected records to JSON

Converts the ESM and NGA-Sub records selected across both AvgSA campaigns
(`esm_records_to_convert.csv` / `ngasub_records_to_convert.csv` from
`wp1pt3pt8i`) into per-record JSON files in `data_processed/07_gm_records`.

The conversion logic lives in the command-line scripts
`convert_esm_records_to_json.py` and `convert_ngasub_records_to_json.py`
(`phd_project/scripts/data_handling/`); here we call their importable
run-functions, which read the records from the network share in parallel.

In [ ]:
from pathlib import Path

import pandas as pd

from phd_project.config import config
from phd_project.scripts.data_handling.convert_esm_records_to_json import (
    convert_esm_records,
)
from phd_project.scripts.data_handling.convert_ngasub_records_to_json import (
    convert_ngasub_records,
)

cfg = config.load_config()

## Paths

In [ ]:
# Source record archives (network share - external resource)
ESM_SRC = cfg["raw_data"]["esm_hdf5_folder"]
NGASUB_SRC = cfg["raw_data"]["ngasub_folder"]

# Combined selection lists (from wp1pt3pt8i)
esm_selection_csv = cfg["proc_data"]["gm_selection"] / "esm_records_to_convert.csv"
ngasub_selection_csv = cfg["proc_data"]["gm_selection"] / "ngasub_records_to_convert.csv"

# NGA-Sub reference flatfiles
ngasub_filename_map = cfg["raw_data"]["gm_flatfiles"] / "NGASub_SA_H1_H2_filenamemap.csv"
ngasub_metadata = cfg["raw_data"]["gm_flatfiles"] / "NGASub_Metadata_SA_rotD50.csv"

# Output folder for the converted JSON records
output_dir = cfg["proc_data"]["gm_records"]

## Convert ESM records

In [ ]:
n_esm, esm_skipped = convert_esm_records(
    ESM_SRC, esm_selection_csv, output_dir)

## Convert NGA-Sub records

In [ ]:
n_ngasub, ngasub_skipped = convert_ngasub_records(
    NGASUB_SRC, ngasub_selection_csv, output_dir,
    filename_map=ngasub_filename_map, metadata=ngasub_metadata)

## Records that could not be converted

Any selected records that were skipped (e.g. missing source file) are listed
below for inspection.

In [ ]:
esm_skipped_df = pd.DataFrame(esm_skipped, columns=["record_identifier", "component", "reason"])
ngasub_skipped_df = pd.DataFrame(ngasub_skipped, columns=["record_identifier", "component", "reason"])

print(f"ESM skipped: {len(esm_skipped_df)} | NGASub skipped: {len(ngasub_skipped_df)}")
display(esm_skipped_df)
display(ngasub_skipped_df)